# Part B — M&A Knowledge Assistant (LangChain + RAG, 70%)

A Retrieval-Augmented Generation (RAG) Q&A system that answers questions **using only the provided M&A Due Diligence Playbook PDF** (`data/MA_Due_Diligence_Playbook.pdf`).

- No internet search, Wikipedia, other PDFs or external knowledge bases are used.
- The vector store is built **only** from this PDF.
- The LLM is instructed to answer **only** from retrieved context and to say when the information is not in the document.

**Stack:** LangChain · `PyPDFLoader` · `RecursiveCharacterTextSplitter` · Google Gemini embeddings · FAISS · Gemini chat model.

**Pipeline:** PDF → load → clean → chunk → embed → FAISS → retrieve top-k chunks → grounded prompt → Gemini → answer + sources.

## 0. Setup

In [1]:
# Install dependencies (skip if you already ran: pip install -r requirements.txt)
# %pip install -q langchain langchain-community langchain-core langchain-text-splitters langchain-google-genai faiss-cpu pypdf pandas

In [2]:
import os
import getpass
import numpy as np
import pandas as pd

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableLambda
from langchain_google_genai import GoogleGenerativeAIEmbeddings, ChatGoogleGenerativeAI

pd.set_option("display.max_colwidth", 120)

PDF_PATH = "data/MA_Due_Diligence_Playbook.pdf"   # the ONLY knowledge source

# Model names — change here if Google renames/retires a model
EMBEDDING_MODEL = "models/gemini-embedding-001"
# gemini-2.5-flash is no longer offered to new API keys; flash-lite is fast and available on the free tier
LLM_MODEL = os.environ.get("LLM_MODEL", "gemini-3.5-flash-lite")

# API key: read from the environment, otherwise ask securely (never hard-code keys in the notebook)
if not os.environ.get("GOOGLE_API_KEY"):
    os.environ["GOOGLE_API_KEY"] = os.environ.get("GEMINI_API_KEY") or getpass.getpass("Enter your Gemini API key: ")

/Users/prabhavijayakumar/Downloads/mini-project-c4-c5/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


/Users/prabhavijayakumar/Downloads/mini-project-c4-c5/.venv/lib/python3.9/site-packages/google/api_core/_python_version_support.py:242: FutureWarning: You are using a non-supported Python version (3.9.6). Google will not post any further updates to google.api_core supporting this Python version. Please upgrade to the latest Python version, or at least Python 3.11, and then update google.api_core.
  warnings.warn(message, FutureWarning)
/Users/prabhavijayakumar/Downloads/mini-project-c4-c5/.venv/lib/python3.9/site-packages/google/auth/__init__.py:54: FutureWarning: You are using a Python version 3.9 past its end of life. Google will update google-auth with critical bug fixes on a best-effort basis, but not with any other fixes or features. Please upgrade your Python version, and then update google-auth.
  warnings.warn(eol_message.format("3.9"), FutureWarning)
/Users/prabhavijayakumar/Downloads/mini-project-c4-c5/.venv/lib/python3.9/site-packages/google/oauth2/__init__.py:40: FutureWarn

/Users/prabhavijayakumar/Downloads/mini-project-c4-c5/.venv/lib/python3.9/site-packages/google/api_core/_python_version_support.py:242: FutureWarning: You are using a non-supported Python version (3.9.6). Google will not post any further updates to google.ai.generativelanguage_v1beta supporting this Python version. Please upgrade to the latest Python version, or at least Python 3.11, and then update google.ai.generativelanguage_v1beta.
  warnings.warn(message, FutureWarning)


## Task 1 — Load and Process the PDF

In [3]:
# 1. Load the M&A PDF with LangChain's PyPDFLoader (one Document per page)
loader = PyPDFLoader(PDF_PATH)
pages = loader.load()
print(f"Loaded {len(pages)} pages from {PDF_PATH}")
for p in pages:
    print(f"  page {p.metadata['page'] + 1}: {len(p.page_content)} characters")

Loaded 2 pages from data/MA_Due_Diligence_Playbook.pdf
  page 1: 2731 characters
  page 2: 808 characters


In [4]:
# 2. Extract the document contents.
# The PDF uses a special glyph (■) for hyphens and renders 'M&A' as 'M&A;' — clean these so the text embeds well.
def clean_text(text: str) -> str:
    return text.replace("\u25a0", "-").replace("M&A;", "M&A").replace("R&W;", "R&W")

for p in pages:
    p.page_content = clean_text(p.page_content)
    p.metadata["source"] = os.path.basename(PDF_PATH)
    p.metadata["page_number"] = p.metadata["page"] + 1   # human-friendly 1-based page

full_text = "\n".join(p.page_content for p in pages)
print(f"Total characters: {len(full_text)}\n")
print(full_text[:1200])

Total characters: 3535

M&A Due Diligence Playbook
Key Clauses, Valuation Methods, and Risk Checklists
Generated: Aug 24, 2025 06:36
A compact reference for consultants. Pairs with the FAISS+OpenAI RAG notebook and the AutoGen
multi-agent pipeline to extract clauses, compute quick valuations, and flag risks from deal documents.
1) Core Clauses in M&A Agreements

Confidentiality / NDA — Mutual confidentiality, carve-outs, survival, destruction/return.

Structure & Consideration — Asset vs stock vs merger; cash/stock mix; WC adjustments; earn-outs;
escrows/holdbacks.

Representations & Warranties — Accuracy of financials; no undisclosed liabilities; compliance; IP;
privacy/cyber; taxes; survival/materiality.

Disclosure Schedules — Qualify reps; list exceptions, contracts, litigation, IP, consents.

Covenants (Pre/Post-Closing) — Ordinary-course ops; no leakage; access; efforts for consents; TSA
scope.

Conditions Precedent — Regulatory/FDI approvals; consents; no MAC; bring-downs.

### 3–4. Chunking — experiment with chunk size and overlap

`RecursiveCharacterTextSplitter` splits on paragraph breaks first, then lines, then sentences/words, so chunks follow the document's natural structure (sections → clause bullets).

The playbook is short (~3,500 characters, 2 pages) and organised as **4 sections**, each a list of short bullets (~100–250 characters each, e.g. *"Indemnification / R&W Insurance — Baskets, caps, escrows…"*). We first compare chunk statistics for several settings, then measure **retrieval quality** for each setting on a small set of test questions with known answers.

In [5]:
def make_chunks(chunk_size, chunk_overlap):
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=["\n\n", "\n", ". ", " ", ""],
        add_start_index=True,
    )
    return splitter.split_documents(pages)

configs = [(150, 0), (150, 30), (300, 50), (500, 50), (500, 100), (800, 100), (1500, 200)]
rows = []
for size, overlap in configs:
    ch = make_chunks(size, overlap)
    lengths = [len(c.page_content) for c in ch]
    rows.append({"chunk_size": size, "overlap": overlap, "num_chunks": len(ch),
                 "avg_len": round(np.mean(lengths)), "min_len": min(lengths), "max_len": max(lengths)})
chunk_stats = pd.DataFrame(rows)
chunk_stats

,chunk_size,overlap,num_chunks,avg_len,min_len,max_len
0,150,0,33,106,65,149
1,150,30,33,111,67,149
2,300,50,16,230,89,290
3,500,50,9,399,89,466
4,500,100,9,441,393,477
5,800,100,6,640,130,789
6,1500,200,3,1228,806,1466


In [6]:
# Look at what a few chunks actually contain for a small and a large setting
for size, overlap in [(150, 30), (1500, 200)]:
    ch = make_chunks(size, overlap)
    print(f"=== chunk_size={size}, overlap={overlap}: {len(ch)} chunks — first 2 shown ===")
    for c in ch[:2]:
        print(repr(c.page_content[:300]), "\n")

=== chunk_size=150, overlap=30: 33 chunks — first 2 shown ===
'M&A Due Diligence Playbook\nKey Clauses, Valuation Methods, and Risk Checklists\nGenerated: Aug 24, 2025 06:36' 

'Generated: Aug 24, 2025 06:36\nA compact reference for consultants. Pairs with the FAISS+OpenAI RAG notebook and the AutoGen' 

=== chunk_size=1500, overlap=200: 3 chunks — first 2 shown ===
'M&A Due Diligence Playbook\nKey Clauses, Valuation Methods, and Risk Checklists\nGenerated: Aug 24, 2025 06:36\nA compact reference for consultants. Pairs with the FAISS+OpenAI RAG notebook and the AutoGen\nmulti-agent pipeline to extract clauses, compute quick valuations, and flag risks from deal docum' 

'\x7f\nRestrictive Covenants — Non-compete, non-solicit, non-disparagement; scope/territory/duration\nreasonableness.\n2) Valuation Methods & Deal Economics\n\x7f\nComps & Precedents — EV/EBITDA, EV/Sales, P/E; sensitivity for growth/margins/leverage.\n\x7f\nDCF — Unlevered FCF, WACC, terminal value; bridge to impli' 



#### Retrieval-quality experiment

For each setting we build a temporary FAISS index and check, for 8 test questions, whether the top-3 retrieved chunks contain the key phrase that answers the question (**hit rate**). We also record how much text is passed to the LLM (**context size**) — smaller is cheaper and less noisy if the hit rate is equal.

In [7]:
# Embedding model (Task 2 explains embeddings in detail)
import time
from langchain_core.embeddings import Embeddings

class CachedRateLimitedEmbeddings(Embeddings):
    """The Gemini free tier allows 100 embedded texts per minute. This wrapper embeds each distinct
    text only once (the experiment re-uses the same chunks/questions) and throttles + retries on 429s."""

    def __init__(self, inner, per_minute=90, batch_size=20):
        self.inner, self.per_minute, self.batch_size = inner, per_minute, batch_size
        self.cache, self.sent = {}, []

    def _call(self, fn, arg, n):
        for attempt in range(6):
            while sum(k for t, k in self.sent if time.time() - t < 60) + n > self.per_minute:
                time.sleep(1)
            self.sent.append((time.time(), n))
            try:
                return fn(arg)
            except Exception as exc:
                if "429" not in str(exc) and "quota" not in str(exc).lower():
                    raise
                time.sleep(20 * (attempt + 1))
        return fn(arg)

    def embed_documents(self, texts):
        missing = list(dict.fromkeys(t for t in texts if ("d", t) not in self.cache))
        for i in range(0, len(missing), self.batch_size):
            batch = missing[i:i + self.batch_size]
            for t, v in zip(batch, self._call(self.inner.embed_documents, batch, len(batch))):
                self.cache[("d", t)] = v
        return [self.cache[("d", t)] for t in texts]

    def embed_query(self, text):
        if ("q", text) not in self.cache:
            self.cache[("q", text)] = self._call(self.inner.embed_query, text, 1)
        return self.cache[("q", text)]

embeddings = CachedRateLimitedEmbeddings(GoogleGenerativeAIEmbeddings(model=EMBEDDING_MODEL))

eval_set = [
    ("What do indemnification provisions cover?", "Baskets, caps, escrows"),
    ("Which valuation multiples are used for comparable companies?", "EV/EBITDA"),
    ("What does a DCF valuation involve?", "WACC"),
    ("What risks fall under cyber and data privacy?", "breach history"),
    ("What should a confidentiality agreement include?", "carve-outs"),
    ("What are typical conditions precedent?", "Regulatory/FDI approvals"),
    ("How should restrictive covenants be assessed?", "Non-compete, non-solicit"),
    ("What are the people and culture risks?", "unions/works councils"),
]

# Embed the questions once and reuse them for every chunk setting
question_vectors = embeddings.embed_documents([q for q, _ in eval_set])

exp_rows = []
for size, overlap in configs:
    ch = make_chunks(size, overlap)
    store = FAISS.from_documents(ch, embeddings, normalize_L2=True)
    hits, ctx = 0, []
    for (q, key), qv in zip(eval_set, question_vectors):
        docs = store.similarity_search_by_vector(qv, k=3)
        hits += any(key.lower() in d.page_content.lower() for d in docs)
        ctx.append(sum(len(d.page_content) for d in docs))
    exp_rows.append({"chunk_size": size, "overlap": overlap, "num_chunks": len(ch),
                     "hit_rate@3": f"{hits}/{len(eval_set)}", "avg_context_chars": int(np.mean(ctx))})

experiment = pd.DataFrame(exp_rows)
experiment

,chunk_size,overlap,num_chunks,hit_rate@3,avg_context_chars
0,150,0,33,8/8,306
1,150,30,33,8/8,312
2,300,50,16,7/8,722
3,500,50,9,8/8,1238
4,500,100,9,8/8,1344
5,800,100,6,8/8,1773
6,1500,200,3,8/8,3685


In [8]:
# Final choice — data-driven selection rule (explained in the report below):
#   consider settings with 300 <= chunk_size <= 800 (smaller chunks cut bullets away from their headings;
#   larger chunks cover almost a whole page and stop being selective),
#   take the highest retrieval hit rate, and break ties by the smallest context passed to the LLM.
candidates = experiment[experiment["chunk_size"].between(300, 800)].copy()
candidates["hits"] = candidates["hit_rate@3"].str.split("/").str[0].astype(int)
best = candidates.sort_values(["hits", "avg_context_chars"], ascending=[False, True]).iloc[0]
CHUNK_SIZE, CHUNK_OVERLAP = int(best["chunk_size"]), int(best["overlap"])
chunks = make_chunks(CHUNK_SIZE, CHUNK_OVERLAP)
for i, c in enumerate(chunks):
    c.metadata["chunk_id"] = i

print(f"Final setting: chunk_size={CHUNK_SIZE}, chunk_overlap={CHUNK_OVERLAP} -> {len(chunks)} chunks\n")
for c in chunks[:3]:
    print(f"[chunk {c.metadata['chunk_id']} | page {c.metadata['page_number']}]\n{c.page_content}\n")

Final setting: chunk_size=500, chunk_overlap=50 -> 9 chunks

[chunk 0 | page 1]
M&A Due Diligence Playbook
Key Clauses, Valuation Methods, and Risk Checklists
Generated: Aug 24, 2025 06:36
A compact reference for consultants. Pairs with the FAISS+OpenAI RAG notebook and the AutoGen
multi-agent pipeline to extract clauses, compute quick valuations, and flag risks from deal documents.
1) Core Clauses in M&A Agreements

Confidentiality / NDA — Mutual confidentiality, carve-outs, survival, destruction/return.


[chunk 1 | page 1]

Structure & Consideration — Asset vs stock vs merger; cash/stock mix; WC adjustments; earn-outs;
escrows/holdbacks.

Representations & Warranties — Accuracy of financials; no undisclosed liabilities; compliance; IP;
privacy/cyber; taxes; survival/materiality.

Disclosure Schedules — Qualify reps; list exceptions, contracts, litigation, IP, consents.

Covenants (Pre/Post-Closing) — Ordinary-course ops; no leakage; access; efforts for consents; TSA
scope.




### 5. Report — Chunking questions

**1. Why is document chunking required?**
- **Embedding/context limits:** embedding models and LLMs have a maximum input size; a long document cannot be embedded or sent to the LLM as one block.
- **Retrieval precision:** one vector for a whole document blurs all its topics together. Smaller chunks each represent one idea, so a question about *indemnification caps* matches the chunk about indemnification, not the whole playbook.
- **Grounded, cheaper answers:** only the few relevant chunks are passed to the LLM, which reduces cost/latency and the noise the model has to read, and makes it possible to cite the exact source passage.

**2. What happens if chunks are too small?** (e.g. 150 characters)
- Chunks lose context: a bullet can be cut in half, separating a heading such as *"Indemnification / R&W Insurance"* from its details (*"baskets, caps, escrows"*).
- Each chunk carries little meaning, so embeddings are less distinctive and retrieval can return fragments that contain the keyword but not the full answer.
- More chunks are needed to answer one question (the answer is scattered), and there is more redundancy from overlap.

**3. What happens if chunks are too large?** (e.g. 1,500 characters)
- Several unrelated topics share one embedding, so the vector is a "blurred average" and similarity scores become less discriminative.
- In this small PDF, 1,500-character chunks cover almost a whole page — retrieval stops being selective and the LLM receives a lot of irrelevant text (higher cost, more room for the model to mix up topics or hallucinate connections).
- Source citations become less useful because each "source" is most of the document.

**4. How did you select the chunk size?**
- **Document structure:** the playbook consists of short clause/risk bullets (~100–250 characters) grouped into four sections of ~700–1,100 characters. That makes **~300–800 characters** the sensible range: big enough to hold 2–4 complete, related bullets, small enough to stay focused on one topic. Chunks of 1,500 characters are almost a whole page (3 chunks for the entire PDF), so retrieval would no longer be selective.
- **Overlap (10–20% of the chunk size)** makes sure a bullet that falls on a chunk boundary appears complete in at least one chunk, so no answer is lost at the edges.
- **Experiment:** the tables above compare chunk counts, the **retrieval hit rate** on 8 questions with known answers, and the **amount of context** sent to the LLM. 150-character chunks also score well on these 8 short questions (each answer is a single bullet), but they split bullets away from their headings (see question 2), so they are excluded. Within the 300–800 range, the notebook selects the setting with the **highest hit rate**, breaking ties by the **smallest context** (cheaper, less noise for the LLM). The chosen values are printed in the "Final setting" output above.

## Task 2 — Generate Embeddings

In [9]:
# Embed all chunks with the Gemini embedding model
chunk_vectors = embeddings.embed_documents([c.page_content for c in chunks])
print(f"Embedded {len(chunk_vectors)} chunks; each embedding has {len(chunk_vectors[0])} dimensions")
print("First 8 values of chunk 0's vector:", np.round(chunk_vectors[0][:8], 4))

Embedded 9 chunks; each embedding has 3072 dimensions
First 8 values of chunk 0's vector: [ 0.0003 -0.0024 -0.0126 -0.0718  0.0083  0.0121  0.0043 -0.0031]


In [10]:
# Demonstrate semantic similarity with cosine similarity
def cosine(a, b):
    a, b = np.array(a), np.array(b)
    with np.errstate(all="ignore"):   # numpy 2.0 + macOS Accelerate raises spurious matmul warnings
        return float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b)))

query = "How is the seller's liability limited after closing?"
q_vec = embeddings.embed_query(query)

sims = pd.DataFrame({
    "chunk_id": [c.metadata["chunk_id"] for c in chunks],
    "page": [c.metadata["page_number"] for c in chunks],
    "cosine_similarity": [round(cosine(q_vec, v), 4) for v in chunk_vectors],
    "text": [c.page_content[:110].replace("\n", " ") + "…" for c in chunks],
}).sort_values("cosine_similarity", ascending=False)

print("Query:", query)
sims.head(5)

Query: How is the seller's liability limited after closing?


,chunk_id,page,cosine_similarity,text
1,1,1,0.7906, Structure & Consideration — Asset vs stock vs merger; cash/stock mix; WC adjustments; earn-outs; escrows/hol…
2,2,1,0.7526,scope.  Conditions Precedent — Regulatory/FDI approvals; consents; no MAC; bring-downs.  Indemnification / R…
5,5,1,0.7492, Legal/Regulatory — Antitrust/FDI; litigation; assignability/CoC; sanctions/export controls.  Operational/In…
3,3,1,0.7419," Restrictive Covenants — Non-compete, non-solicit, non-disparagement; scope/territory/duration reasonableness…"
6,6,1,0.7363, Macro/Financing — Rate sensitivity; FX/commodity; financing certainty; MAC; break-fees.…


Note that the query uses the words *"seller's liability limited"*, which do **not** appear in the PDF, yet the top results include the *Indemnification — baskets, caps, escrows* chunk. That is semantic (meaning-based) matching rather than keyword matching.

### Explanations — Task 2

**1. What is an embedding?**
An embedding is a **dense numerical vector** (here a few thousand floating-point numbers) produced by a neural network that represents the *meaning* of a piece of text. Texts with similar meaning are mapped to vectors that are close together in this high-dimensional space; unrelated texts are far apart. For example, "indemnity cap" and "limit on the seller's liability" get similar vectors even though they share few words.

**2. Why are embeddings required for RAG?**
RAG must find, among all chunks, the few that are relevant to a user's question — quickly and by **meaning**, not just exact words. Embeddings turn both the chunks and the question into vectors in the same space, so relevance becomes a fast mathematical distance calculation that a vector store (FAISS) can perform over thousands or millions of chunks. Without embeddings we would be limited to keyword search, which misses synonyms and paraphrases (e.g. "acquisition price" vs "consideration").

**3. How does semantic similarity work in retrieval?**
1. **Offline:** every chunk is embedded once and stored in the vector index.
2. **At question time:** the question is embedded with the **same** model.
3. The vector store compares the question vector with every chunk vector using a similarity measure — typically **cosine similarity** (the cosine of the angle between two vectors: 1 = same direction/meaning, ≈0 = unrelated) or the equivalent Euclidean distance on normalised vectors.
4. The **top-k** closest chunks are returned and passed to the LLM as context.

Because the vectors encode meaning, a chunk can be retrieved even when it uses different wording from the question, as shown above.

## Task 3 — Build a Vector Store (FAISS)

In [11]:
# Build the FAISS index from the M&A chunks only (re-using the embeddings computed above)
vector_store = FAISS.from_embeddings(
    text_embeddings=list(zip([c.page_content for c in chunks], chunk_vectors)),
    embedding=embeddings,
    metadatas=[c.metadata for c in chunks],
    normalize_L2=True,   # unit-length vectors -> L2 distance is equivalent to cosine similarity
)

# Verify the store contains ONLY information from the provided M&A PDF
stored_docs = list(vector_store.docstore._dict.values())
sources = {d.metadata["source"] for d in stored_docs}
print(f"Vectors in index: {vector_store.index.ntotal}")
print(f"Documents in docstore: {len(stored_docs)}")
print(f"Distinct sources: {sources}")
assert sources == {os.path.basename(PDF_PATH)}, "Vector store contains non-M&A content!"

# Persist the index so it can be reloaded without re-embedding
vector_store.save_local("faiss_index_ma_playbook")
print("Saved FAISS index to ./faiss_index_ma_playbook")

Vectors in index: 9
Documents in docstore: 9
Distinct sources: {'MA_Due_Diligence_Playbook.pdf'}
Saved FAISS index to ./faiss_index_ma_playbook


In [12]:
# Quick retrieval check.
# With unit-length vectors FAISS returns the squared L2 distance d², and cosine similarity = 1 - d²/2.
def search_with_cosine(question, k):
    return [(doc, 1 - float(d2) / 2) for doc, d2 in vector_store.similarity_search_with_score(question, k=k)]

for doc, score in search_with_cosine("What valuation methods are covered?", k=3):
    print(f"[page {doc.metadata['page_number']} | chunk {doc.metadata['chunk_id']} | cosine similarity {score:.3f}]")
    print(doc.page_content[:200].replace("\n", " "), "…\n")

[page 1 | chunk 3 | cosine similarity 0.811]
 Restrictive Covenants — Non-compete, non-solicit, non-disparagement; scope/territory/duration reasonableness. 2) Valuation Methods & Deal Economics  Comps & Precedents — EV/EBITDA, EV/Sales, P/E; s …

[page 1 | chunk 6 | cosine similarity 0.782]
 Macro/Financing — Rate sensitivity; FX/commodity; financing certainty; MAC; break-fees. …

[page 1 | chunk 0 | cosine similarity 0.764]
M&A Due Diligence Playbook Key Clauses, Valuation Methods, and Risk Checklists Generated: Aug 24, 2025 06:36 A compact reference for consultants. Pairs with the FAISS+OpenAI RAG notebook and the AutoG …



## Task 4 & Task 5 — RAG Pipeline with Grounded Prompt

The prompt (Task 5) instructs the LLM to:
- use **only** the retrieved context,
- **not hallucinate** or use general knowledge,
- **clearly say** when the answer is not in the M&A Playbook,
- give a **concise but complete** answer, citing page numbers.

In [13]:
NOT_FOUND_MSG = "The information is not available in the provided M&A Playbook."

SYSTEM_PROMPT = f"""You are an M&A Knowledge Assistant. You answer questions using ONLY the context
retrieved from the organisation's "M&A Due Diligence Playbook" PDF.

Rules:
1. Use ONLY the information in the CONTEXT below. Do not use general knowledge, the internet, or any other source.
2. Do NOT hallucinate: never invent facts, numbers, names, examples, case studies or details that are not explicitly in the context.
3. If the context does not contain the answer, reply exactly: "{NOT_FOUND_MSG}"
   If only part of the question is answered by the context, answer that part and state which part is not covered in the playbook.
4. Be concise but complete: cover every relevant point from the context, using short bullet points where helpful.
5. Cite the page number(s) you used in the form [p.X]."""

rag_prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT),
    ("human", "CONTEXT (retrieved from the M&A Playbook):\n{context}\n\nQUESTION: {question}\n\nANSWER:"),
])

llm = ChatGoogleGenerativeAI(model=LLM_MODEL, temperature=0)   # temperature 0 -> factual, deterministic answers

TOP_K = 4

def retrieve(question: str):
    """Retrieve the top-k most similar chunks (with cosine similarity scores) from FAISS."""
    return search_with_cosine(question, k=TOP_K)

def format_context(results) -> str:
    return "\n\n".join(f"[p.{d.metadata['page_number']}] {d.page_content}" for d, _ in results)

# LCEL chain: question -> retrieve -> format context -> prompt -> LLM -> text
rag_chain = (
    {"results": RunnableLambda(retrieve), "question": RunnablePassthrough()}
    | RunnablePassthrough.assign(context=lambda x: format_context(x["results"]))
    | RunnablePassthrough.assign(answer=rag_prompt | llm | StrOutputParser())
)
print(rag_prompt.messages[0].prompt.template)

You are an M&A Knowledge Assistant. You answer questions using ONLY the context
retrieved from the organisation's "M&A Due Diligence Playbook" PDF.

Rules:
1. Use ONLY the information in the CONTEXT below. Do not use general knowledge, the internet, or any other source.
2. Do NOT hallucinate: never invent facts, numbers, names, examples, case studies or details that are not explicitly in the context.
3. If the context does not contain the answer, reply exactly: "The information is not available in the provided M&A Playbook."
   If only part of the question is answered by the context, answer that part and state which part is not covered in the playbook.
4. Be concise but complete: cover every relevant point from the context, using short bullet points where helpful.
5. Cite the page number(s) you used in the form [p.X].


In [14]:
def ask(question: str, show_sources: bool = True):
    """1. accept a question  2. retrieve chunks  3. pass context to the LLM
       4. generate an answer  5. display the answer  6. display the retrieved sources"""
    out = rag_chain.invoke(question)
    print("=" * 100)
    print("QUESTION:", question)
    print("-" * 100)
    print("ANSWER:\n" + out["answer"].strip())
    if show_sources:
        print("-" * 100)
        print("RETRIEVED CONTEXT / SOURCES:")
        for doc, score in out["results"]:
            snippet = doc.page_content.replace("\n", " ")
            print(f"  • {doc.metadata['source']} | page {doc.metadata['page_number']} | "
                  f"chunk {doc.metadata['chunk_id']} | cosine similarity {score:.3f}")
            print(f"    \"{snippet[:300]}{'…' if len(snippet) > 300 else ''}\"")
    print("=" * 100, "\n")
    return out

### Demo — questions answered from the playbook

In [15]:
in_scope_questions = [
    "What does the Indemnification / R&W Insurance clause cover?",
    "Which valuation methods does the playbook describe?",
    "What should a Confidentiality / NDA clause contain?",
    "What are the key cyber and data privacy risks, and how can they be mitigated?",
    "What are the conditions precedent to closing?",
    "Which financial and accounting risks should be checked during due diligence?",
]
for q in in_scope_questions:
    ask(q)

QUESTION: What does the Indemnification / R&W Insurance clause cover?
----------------------------------------------------------------------------------------------------
ANSWER:
Based on the M&A Due Diligence Playbook, the Indemnification / R&W Insurance clause covers:

* Baskets, caps, and escrows [p.1]
* Special indemnities (tax/litigation) [p.1]
* RWI (Representations & Warranties Insurance) retentions and exclusions [p.1]
----------------------------------------------------------------------------------------------------
RETRIEVED CONTEXT / SOURCES:
  • MA_Due_Diligence_Playbook.pdf | page 1 | chunk 2 | cosine similarity 0.812
    "scope.  Conditions Precedent — Regulatory/FDI approvals; consents; no MAC; bring-downs.  Indemnification / R&W Insurance — Baskets, caps, escrows; special indemnities (tax/litigation); RWI retentions/exclusions.  Termination & Fees — Outside date; termination for breach/failure of conditions; rev…"
  • MA_Due_Diligence_Playbook.pdf | page 1 | chunk 1

QUESTION: Which valuation methods does the playbook describe?
----------------------------------------------------------------------------------------------------
ANSWER:
Based on the provided M&A Playbook, the valuation methods described are:

* **Comps & Precedents** — EV/EBITDA, EV/Sales, P/E; sensitivity for growth/margins/leverage [p.1].
* **DCF** — Unlevered FCF, WACC, terminal value; bridge to implied multiples; scenarios [p.1].
* **Asset / SOTP** — Fair-value assets/segments where balance sheet or IP drives value [p.1].
----------------------------------------------------------------------------------------------------
RETRIEVED CONTEXT / SOURCES:
  • MA_Due_Diligence_Playbook.pdf | page 1 | chunk 3 | cosine similarity 0.800
    " Restrictive Covenants — Non-compete, non-solicit, non-disparagement; scope/territory/duration reasonableness. 2) Valuation Methods & Deal Economics  Comps & Precedents — EV/EBITDA, EV/Sales, P/E; sensitivity for growth/margins/leverage.  DCF — Unle

QUESTION: What should a Confidentiality / NDA clause contain?
----------------------------------------------------------------------------------------------------
ANSWER:
Based on the provided M&A Playbook, a Confidentiality / NDA clause should contain:

* Mutual confidentiality [p.1]
* Carve-outs [p.1]
* Survival terms [p.1]
* Destruction or return provisions [p.1]
----------------------------------------------------------------------------------------------------
RETRIEVED CONTEXT / SOURCES:
  • MA_Due_Diligence_Playbook.pdf | page 1 | chunk 1 | cosine similarity 0.819
    " Structure & Consideration — Asset vs stock vs merger; cash/stock mix; WC adjustments; earn-outs; escrows/holdbacks.  Representations & Warranties — Accuracy of financials; no undisclosed liabilities; compliance; IP; privacy/cyber; taxes; survival/materiality.  Disclosure Schedules — Qualify reps…"
  • MA_Due_Diligence_Playbook.pdf | page 1 | chunk 3 | cosine similarity 0.793
    " Restrictive Covenants — Non-

QUESTION: What are the key cyber and data privacy risks, and how can they be mitigated?
----------------------------------------------------------------------------------------------------
ANSWER:
Based on the provided M&A Playbook:

**Key cyber and data privacy risks:**
* Security posture [p.1]
* Breach history [p.1]
* GDPR/CCPA compliance [p.1]

**Mitigations:**
* Cyber reps & special indemnities [p.1]
----------------------------------------------------------------------------------------------------
RETRIEVED CONTEXT / SOURCES:
  • MA_Due_Diligence_Playbook.pdf | page 1 | chunk 5 | cosine similarity 0.795
    " Legal/Regulatory — Antitrust/FDI; litigation; assignability/CoC; sanctions/export controls.  Operational/Integration — IT/ERP fit; data migration; supply chain; Day-1/TSA; PMO cadence.  People/Culture — Retention; incentives; unions/works councils; culture diagnostics & comms.  Cyber/Data Priva…"
  • MA_Due_Diligence_Playbook.pdf | page 1 | chunk 1 | cosine similarity 0.

QUESTION: What are the conditions precedent to closing?
----------------------------------------------------------------------------------------------------
ANSWER:
Based on the provided M&A Playbook, the conditions precedent to closing are:

* Regulatory/FDI approvals [p.1]
* Consents [p.1]
* No MAC (Material Adverse Change) [p.1]
* Bring-downs (of representations and warranties) [p.1]
----------------------------------------------------------------------------------------------------
RETRIEVED CONTEXT / SOURCES:
  • MA_Due_Diligence_Playbook.pdf | page 1 | chunk 1 | cosine similarity 0.841
    " Structure & Consideration — Asset vs stock vs merger; cash/stock mix; WC adjustments; earn-outs; escrows/holdbacks.  Representations & Warranties — Accuracy of financials; no undisclosed liabilities; compliance; IP; privacy/cyber; taxes; survival/materiality.  Disclosure Schedules — Qualify reps…"
  • MA_Due_Diligence_Playbook.pdf | page 1 | chunk 2 | cosine similarity 0.820
    "scope.  

QUESTION: Which financial and accounting risks should be checked during due diligence?
----------------------------------------------------------------------------------------------------
ANSWER:
Based on the provided M&A Playbook, the financial and accounting risks to be checked during due diligence are:

* Quality of Earnings (QoE) [p.1]
* Working Capital (WC) seasonality [p.1]
* Off-balance sheet (off-BS) liabilities [p.1]
* Tax exposures [p.1]
* Covenants [p.1]

[p.1]
----------------------------------------------------------------------------------------------------
RETRIEVED CONTEXT / SOURCES:
  • MA_Due_Diligence_Playbook.pdf | page 1 | chunk 5 | cosine similarity 0.823
    " Legal/Regulatory — Antitrust/FDI; litigation; assignability/CoC; sanctions/export controls.  Operational/Integration — IT/ERP fit; data migration; supply chain; Day-1/TSA; PMO cadence.  People/Culture — Retention; incentives; unions/works councils; culture diagnostics & comms.  Cyber/Data Priva…"
  • MA

### Demo — hallucination guard (questions NOT answered by the playbook)

The brief's topic list mentions a *TargetCo case study* and detailed *post-merger integration*, but this PDF does not contain them. A grounded system must say so instead of inventing an answer from general knowledge.

In [16]:
out_of_scope_questions = [
    "What happened in the TargetCo case study?",
    "What was the purchase price paid for TargetCo?",
    "Who is the CEO of Microsoft?",
    "What is the typical percentage cap on indemnification claims in US deals?",
]
for q in out_of_scope_questions:
    ask(q)

QUESTION: What happened in the TargetCo case study?
----------------------------------------------------------------------------------------------------
ANSWER:
The information is not available in the provided M&A Playbook.
----------------------------------------------------------------------------------------------------
RETRIEVED CONTEXT / SOURCES:
  • MA_Due_Diligence_Playbook.pdf | page 2 | chunk 8 | cosine similarity 0.736
    "TSA, retention). Cite [p.X]. References  Bain & Company – Global M&A Report 2025 (PDF): https://www.bain.com/globalassets/noindex/2025/bain_report_global_m_and_a_report_2025.pdf  Bain – M&A Report Topic Hub: https://www.bain.com/insights/topics/m-and-a-report/  DLA Piper – Understanding Basic Dea…"
  • MA_Due_Diligence_Playbook.pdf | page 1 | chunk 3 | cosine similarity 0.734
    " Restrictive Covenants — Non-compete, non-solicit, non-disparagement; scope/territory/duration reasonableness. 2) Valuation Methods & Deal Economics  Comps & Precedents — EV

QUESTION: What was the purchase price paid for TargetCo?
----------------------------------------------------------------------------------------------------
ANSWER:
The information is not available in the provided M&A Playbook.
----------------------------------------------------------------------------------------------------
RETRIEVED CONTEXT / SOURCES:
  • MA_Due_Diligence_Playbook.pdf | page 1 | chunk 1 | cosine similarity 0.810
    " Structure & Consideration — Asset vs stock vs merger; cash/stock mix; WC adjustments; earn-outs; escrows/holdbacks.  Representations & Warranties — Accuracy of financials; no undisclosed liabilities; compliance; IP; privacy/cyber; taxes; survival/materiality.  Disclosure Schedules — Qualify reps…"
  • MA_Due_Diligence_Playbook.pdf | page 1 | chunk 3 | cosine similarity 0.797
    " Restrictive Covenants — Non-compete, non-solicit, non-disparagement; scope/territory/duration reasonableness. 2) Valuation Methods & Deal Economics  Comps & Precedents

QUESTION: Who is the CEO of Microsoft?
----------------------------------------------------------------------------------------------------
ANSWER:
The information is not available in the provided M&A Playbook.
----------------------------------------------------------------------------------------------------
RETRIEVED CONTEXT / SOURCES:
  • MA_Due_Diligence_Playbook.pdf | page 1 | chunk 0 | cosine similarity 0.703
    "M&A Due Diligence Playbook Key Clauses, Valuation Methods, and Risk Checklists Generated: Aug 24, 2025 06:36 A compact reference for consultants. Pairs with the FAISS+OpenAI RAG notebook and the AutoGen multi-agent pipeline to extract clauses, compute quick valuations, and flag risks from deal docum…"
  • MA_Due_Diligence_Playbook.pdf | page 1 | chunk 3 | cosine similarity 0.702
    " Restrictive Covenants — Non-compete, non-solicit, non-disparagement; scope/territory/duration reasonableness. 2) Valuation Methods & Deal Economics  Comps & Precedents — EV/EBITDA, EV/S

QUESTION: What is the typical percentage cap on indemnification claims in US deals?
----------------------------------------------------------------------------------------------------
ANSWER:
The information is not available in the provided M&A Playbook.
----------------------------------------------------------------------------------------------------
RETRIEVED CONTEXT / SOURCES:
  • MA_Due_Diligence_Playbook.pdf | page 1 | chunk 1 | cosine similarity 0.840
    " Structure & Consideration — Asset vs stock vs merger; cash/stock mix; WC adjustments; earn-outs; escrows/holdbacks.  Representations & Warranties — Accuracy of financials; no undisclosed liabilities; compliance; IP; privacy/cyber; taxes; survival/materiality.  Disclosure Schedules — Qualify reps…"
  • MA_Due_Diligence_Playbook.pdf | page 1 | chunk 2 | cosine similarity 0.831
    "scope.  Conditions Precedent — Regulatory/FDI approvals; consents; no MAC; bring-downs.  Indemnification / R&W Insurance — Baskets, caps, esc

### Ask your own question

In [17]:
# Change the question and re-run this cell
ask("What is an earn-out and where does it appear in the playbook?")

QUESTION: What is an earn-out and where does it appear in the playbook?
----------------------------------------------------------------------------------------------------
ANSWER:
Based on the provided playbook:

* **Where it appears:** Earn-outs appear in two sections on page 1 [p.1]:
  * Under **Structure & Consideration** (as a deal consideration mechanism).
  * Under **Synergies** (noted as "earn-out linkages").
* **Definition/Details:** The provided playbook context mentions "earn-outs" under deal structure and synergies, but it does not provide a formal definition of what an earn-out is [p.1].
----------------------------------------------------------------------------------------------------
RETRIEVED CONTEXT / SOURCES:
  • MA_Due_Diligence_Playbook.pdf | page 1 | chunk 3 | cosine similarity 0.796
    " Restrictive Covenants — Non-compete, non-solicit, non-disparagement; scope/territory/duration reasonableness. 2) Valuation Methods & Deal Economics  Comps & Precedents — EV/EB

{'results': [(Document(id='a3321df6-0f3d-46c9-a7e2-9f63abb5fb31', metadata={'producer': 'ReportLab PDF Library - www.reportlab.com', 'creator': '(unspecified)', 'creationdate': '2025-08-24T06:36:20+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2025-08-24T06:36:20+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': 'MA_Due_Diligence_Playbook.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1', 'page_number': 1, 'start_index': 1315, 'chunk_id': 3}, page_content='\x7f\nRestrictive Covenants — Non-compete, non-solicit, non-disparagement; scope/territory/duration\nreasonableness.\n2) Valuation Methods & Deal Economics\n\x7f\nComps & Precedents — EV/EBITDA, EV/Sales, P/E; sensitivity for growth/margins/leverage.\n\x7f\nDCF — Unlevered FCF, WACC, terminal value; bridge to implied multiples; scenarios.\n\x7f\nAsset / SOTP — Fair-value assets/segments where balance sheet or IP drives value.\n\x7f'),
   0.7961346060037613),
  (Document(id='1

In [18]:
# Optional interactive loop — set INTERACTIVE = True to type questions (empty line to stop)
INTERACTIVE = False
while INTERACTIVE:
    q = input("Ask the M&A assistant (blank to quit): ").strip()
    if not q:
        break
    ask(q)

## Summary — Part B

| Requirement | Implementation |
|---|---|
| Load & process PDF | `PyPDFLoader` → text cleaning → `RecursiveCharacterTextSplitter` (size/overlap chosen by the retrieval experiment) |
| Embeddings | Google Gemini `gemini-embedding-001` |
| Vector store | FAISS, built only from the M&A PDF (verified with an assertion), saved to `faiss_index_ma_playbook/` |
| RAG pipeline | LangChain LCEL: question → FAISS top-k retrieval → grounded prompt → Gemini (`temperature=0`) → answer |
| Sources | Every answer prints the retrieved chunks with page number, chunk id and cosine similarity |
| Prompt engineering | Context-only answers, no hallucination, explicit "not available in the provided M&A Playbook" message, concise answers with [p.X] citations |
| Hallucination test | Out-of-scope questions (TargetCo case study, CEO of Microsoft, specific cap percentages) are refused |